# Gỡ lỗi PPO hỏng mà không báo lỗi

Notebook này là một bản PPO cho CartPole-v1, cùng cấu trúc với bài Intermediate 02: actor-critic, GAE, rollout trên 4 env
song song, clipped objective. Nó chạy hết không một lỗi, đường cong return có lên, nhưng có **5 bug im lặng** trong các cell
định nghĩa, mỗi bug một chỗ. Đề không nói chúng ở đâu.

- Chấm: `uv run pytest tracks/ai-engineer/modules/07-reinforcement-learning/exercises/advanced/02-debug-broken-ppo -q`
- Ghi lại từng bug vào `BUGS.md` cùng thư mục.

In [ ]:
# explore: giới hạn luồng, tìm gốc repo
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn

## 1. Model

In [ ]:
def layer_init(layer: nn.Linear, std: float = 2 ** 0.5) -> nn.Linear:
    nn.init.orthogonal_(layer.weight, std)
    nn.init.zeros_(layer.bias)
    return layer


class ActorCritic(nn.Module):
    """Actor và critic là hai MLP 64-64 tanh tách rời. dist(obs) -> Categorical, value(obs) -> V(s)."""

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        self.actor = nn.Sequential(layer_init(nn.Linear(obs_dim, hidden)), nn.Tanh(),
                                   layer_init(nn.Linear(hidden, hidden)), nn.Tanh(),
                                   layer_init(nn.Linear(hidden, n_actions), std=0.01))
        self.critic = nn.Sequential(layer_init(nn.Linear(obs_dim, hidden)), nn.Tanh(),
                                    layer_init(nn.Linear(hidden, hidden)), nn.Tanh(),
                                    layer_init(nn.Linear(hidden, 1), std=1.0))

    def dist(self, obs: torch.Tensor) -> torch.distributions.Categorical:
        return torch.distributions.Categorical(logits=self.actor(obs))

    def value(self, obs: torch.Tensor) -> torch.Tensor:
        return self.critic(obs)

## 2. Loss và GAE

In [ ]:
def ppo_policy_loss(new_logp: torch.Tensor, old_logp: torch.Tensor, adv: torch.Tensor,
                    clip_eps: float) -> tuple[torch.Tensor, dict]:
    """Clipped surrogate của PPO (dạng loss để tối thiểu hoá) và hai số chẩn đoán."""
    log_ratio = new_logp - old_logp
    ratio = log_ratio.exp()
    loss = -torch.max(ratio * adv, ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv).mean()
    with torch.no_grad():
        clipfrac = ((ratio - 1).abs() > clip_eps).float().mean().item()
        approx_kl = ((ratio - 1) - log_ratio).mean().item()
    return loss, {"clipfrac": clipfrac, "approx_kl": approx_kl}


def gae(rewards, values, next_values, terminated, truncated, gamma, lam):
    """GAE(λ) theo quy ước buffer của bài Basic 03. Trả (advantages, returns)."""
    rewards = np.asarray(rewards, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    next_values = np.asarray(next_values, dtype=np.float64)
    not_term = 1.0 - np.asarray(terminated, dtype=np.float64)
    not_done = not_term * (1.0 - np.asarray(truncated, dtype=np.float64))
    adv = np.zeros_like(rewards)
    running = np.zeros_like(rewards[0])
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * not_done[t] * next_values[t] - values[t]
        running = delta + gamma * lam * not_done[t] * running
        adv[t] = running
    return adv, adv + values

## 3. Thu rollout

In [ ]:
def make_vec_env(make_env, n_envs: int) -> gym.vector.SyncVectorEnv:
    return gym.vector.SyncVectorEnv([make_env] * n_envs, autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)


def collect_rollout(envs, model: ActorCritic, obs: np.ndarray, n_steps: int) -> tuple[dict, np.ndarray]:
    """n_steps bước trên N env. buf: obs, actions, logp, rewards, values, next_values, terminated, truncated (T, N),
    episode_returns (tổng reward các episode kết thúc trong rollout). next_values là V của observation thật sau bước."""
    n = envs.num_envs
    if not hasattr(envs, "ep_returns"):
        envs.ep_returns = np.zeros(n)
    buf = {
        "obs": np.zeros((n_steps, n, obs.shape[1]), dtype=np.float32),
        "actions": np.zeros((n_steps, n), dtype=np.int64),
        "logp": np.zeros((n_steps, n), dtype=np.float32),
        "rewards": np.zeros((n_steps, n)),
        "values": np.zeros((n_steps, n)),
        "next_values": np.zeros((n_steps, n)),
        "terminated": np.zeros((n_steps, n), dtype=bool),
        "truncated": np.zeros((n_steps, n), dtype=bool),
        "episode_returns": [],
    }
    for t in range(n_steps):
        with torch.no_grad():
            o = torch.as_tensor(obs, dtype=torch.float32)
            d = model.dist(o)
            a = d.sample()
            buf["logp"][t] = d.log_prob(a).numpy()
            buf["values"][t] = model.value(o).numpy().reshape(-1)
        next_obs, reward, terminated, truncated, info = envs.step(a.numpy())
        buf["obs"][t], buf["actions"][t], buf["rewards"][t] = obs, a.numpy(), reward
        buf["terminated"][t], buf["truncated"][t] = terminated, truncated
        real_next = np.array(next_obs, dtype=np.float32)
        done = np.logical_or(terminated, truncated)
        for i in np.flatnonzero(done):
            real_next[i] = info["final_obs"][i]
        with torch.no_grad():
            buf["next_values"][t] = model.value(torch.as_tensor(real_next)).numpy().reshape(-1)
        envs.ep_returns += reward
        for i in np.flatnonzero(done):
            buf["episode_returns"].append(float(envs.ep_returns[i]))
        obs = next_obs
    return buf, obs

## 4. Cập nhật và vòng train

In [ ]:
def ppo_update(model: ActorCritic, opt: torch.optim.Optimizer, batch: dict, rng: np.random.Generator, epochs: int = 4,
               n_minibatches: int = 4, clip_eps: float = 0.2, vf_coef: float = 0.5, ent_coef: float = 0.01,
               max_grad_norm: float = 0.5) -> dict:
    """K epoch trên batch đã làm phẳng (obs, actions, logp, advantages, returns). Advantage chuẩn hoá trong minibatch."""
    B = len(batch["obs"])
    size = B // n_minibatches
    for _ in range(epochs):
        perm = rng.permutation(B)
        for start in range(0, B, size):
            idx = torch.as_tensor(perm[start:start + size])
            d = model.dist(batch["obs"][idx])
            adv = batch["advantages"][idx]
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)
            policy_loss, stats = ppo_policy_loss(d.log_prob(batch["actions"][idx]), batch["logp"][idx], adv, clip_eps)
            value_loss = 0.5 * ((model.value(batch["obs"][idx]) - batch["returns"][idx]) ** 2).mean()
            entropy = d.entropy().mean()
            loss = policy_loss + vf_coef * value_loss + ent_coef * entropy
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()
    return {**stats, "entropy": entropy.item(), "value_loss": value_loss.item()}


def train_ppo(make_env, total_steps: int = 150_000, seed: int = 0, n_envs: int = 4, n_steps: int = 128,
              lr: float = 2.5e-4, gamma: float = 0.99, lam: float = 0.95, **update_kwargs) -> dict:
    """PPO đầy đủ. Trả returns, end_steps (tổng bước env ở cuối rollout có episode kết thúc), stats, model."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    envs = make_vec_env(make_env, n_envs)
    obs, _ = envs.reset(seed=seed)
    model = ActorCritic(envs.single_observation_space.shape[0], envs.single_action_space.n)
    opt = torch.optim.Adam(model.parameters(), lr=lr, eps=1e-5)
    n_iters = total_steps // (n_envs * n_steps)
    returns, end_steps, history, step = [], [], [], 0
    for k in range(n_iters):
        opt.param_groups[0]["lr"] = lr * (1 - k / n_iters)
        buf, obs = collect_rollout(envs, model, obs, n_steps)
        step += n_envs * n_steps
        returns += buf["episode_returns"]
        end_steps += [step] * len(buf["episode_returns"])
        adv, ret = gae(buf["rewards"], buf["values"], buf["next_values"], buf["terminated"], buf["truncated"],
                       gamma, lam)
        flat = lambda x, dtype: torch.as_tensor(np.asarray(x).reshape(n_envs * n_steps, *np.shape(x)[2:]), dtype=dtype)
        batch = {"obs": flat(buf["obs"], torch.float32), "actions": flat(buf["actions"], torch.int64),
                 "logp": flat(buf["logp"], torch.float32), "advantages": flat(adv, torch.float32),
                 "returns": flat(ret, torch.float32)}
        history.append({"step": step, **ppo_update(model, opt, batch, rng, **update_kwargs)})
    envs.close()
    return {"returns": returns, "end_steps": end_steps, "stats": history, "model": model}

In [ ]:
# explore: train trên CartPole, 3 seed x 150 000 bước (mỗi seed khoảng 40 giây trên CPU 4 luồng)
import matplotlib.pyplot as plt

torch.set_num_threads(4)
make_env = lambda: gym.make("CartPole-v1")
runs = {}
for seed in [0, 1, 2]:
    start = time.perf_counter()
    runs[seed] = train_ppo(make_env, seed=seed)
    r = np.array(runs[seed]["returns"])
    print(f"seed {seed}: {time.perf_counter() - start:.0f} s | {len(r)} episode | TB 20 episode cuối {r[-20:].mean():.1f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 3.2))
for seed, h in runs.items():
    axes[0].plot(h["end_steps"], np.convolve(h["returns"], np.ones(20) / 20, mode="same"), label=f"seed {seed}")
    for ax, key in zip(axes[1:], ["approx_kl", "clipfrac", "entropy"]):
        ax.plot([s["step"] for s in h["stats"]], [s[key] for s in h["stats"]])
for ax, title in zip(axes, ["return (TB trượt 20)", "approx_kl", "clipfrac", "entropy"]):
    ax.set_title(title); ax.set_xlabel("bước env")
axes[0].legend()
plt.show()

In [ ]:
# explore: policy greedy cuối trên 20 episode cố định (seed 10 000..10 019)
def greedy_return(model, n=20, seed=10_000):
    env, out = make_env(), []
    for i in range(n):
        obs, _ = env.reset(seed=seed + i)
        done, total = False, 0.0
        while not done:
            with torch.no_grad():
                a = int(model.dist(torch.as_tensor(obs).unsqueeze(0)).probs.argmax())
            obs, r, te, tr, _ = env.step(a)
            total += r
            done = te or tr
        out.append(total)
    return float(np.mean(out))

print("greedy, 20 episode:", [round(greedy_return(runs[s]["model"]), 1) for s in [0, 1, 2]])

## Ghi chép

Thêm cell `explore` tuỳ ý cho các phép thử của bạn. Mỗi giả thuyết một phép đo; sửa ở gốc, không đổi siêu tham số để bù triệu
chứng. Sau khi sửa xong, chạy lại cell train và so với bảng PPO của mục 5 lý thuyết.